CREATION OF ELEMENT DATABASE   

In [2]:
import requests
from bs4 import BeautifulSoup
from collections import defaultdict
# Target URL
URL = "http://abulafia.mt.ic.ac.uk/shannon/radius.php"

# Send request
response = requests.get(URL)
response.raise_for_status()

# Parse HTML
soup = BeautifulSoup(response.text, "lxml")

# Find the table (main ionic radii table)
table = soup.find("table")


# element → charge → coordination → radius
elements = dict()

rows = table.find_all("tr") # skip header

for row in rows:

    cols = [c.get_text(strip=True) for c in row.find_all("td")]
    
    if cols==['Ion', 'Charge', 'Coordination', 'Spin State', 'Crystal Radius', 'Ionic Radius', 'Key*']:
        continue

    # Ion | Charge | Coordination | Spin | Crystal R | Ionic R | Key
    if len(cols) > 6:
        element = cols[0]
        charge = cols[1]
        coordination = cols[2]
        ionic_radius = float(cols[5])

        elements[element]={"Charge":{charge:{coordination:ionic_radius}}}

    elif len(cols)>5:
        charge = cols[0]
        coordination = cols[1]
        ionic_radius = float(cols[4])
        elements[element]["Charge"][charge]={coordination:ionic_radius}
    elif len(cols)==5:
        coordination = cols[0]
        ionic_radius = float(cols[3])
        elements[element]["Charge"][charge][coordination]=ionic_radius
    else:
        continue

import pandas as pd

# Load PubChem periodic table
url = "https://pubchem.ncbi.nlm.nih.gov/rest/pug/periodictable/CSV"
atom_info = pd.read_csv(url)

# Iterate over ALL elements from the website
for _, row in atom_info.iterrows():
    symbol = row["Symbol"]

    # If element not in dictionary, create it
    if symbol not in elements:
        elements[symbol] = {}

    # Add / update atomic properties
    elements[symbol]["AtomicNumber"] = int(row["AtomicNumber"])
    elements[symbol]["AtomicMass"] = float(row["AtomicMass"])

print(elements["Ag"])

{'Charge': {'1': {'II': 0.67, 'IV': 1.0, 'IVSQ': 1.02, 'V': 1.09, 'VI': 1.15, 'VII': 1.22, 'VIII': 1.28}, '2': {'IVSQ': 0.79, 'VI': 0.94}, '3': {'IVSQ': 0.67, 'VI': 0.75}}, 'AtomicNumber': 47, 'AtomicMass': 107.868}


MULTITASK NEURAL NETWORK MODEL TRAINING

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score
import warnings

# ==========================================
# 1. ELEMENTAL & ORGANIC PROPERTY DATABASE
# ==========================================
ELEMENT_PROPS = {
    'H': (0.37, 2.20), 'Li': (1.34, 0.98), 'Be': (0.90, 1.57), 'B': (0.82, 2.04), 'C': (0.77, 2.55),
    'N': (0.75, 3.04), 'O': (1.40, 3.44), 'F': (1.33, 3.98), 'Na': (1.54, 0.93), 'Mg': (1.30, 1.31),
    'Al': (1.18, 1.61), 'Si': (1.11, 1.90), 'P': (1.06, 2.19), 'S': (1.02, 2.58), 'Cl': (1.81, 3.16),
    'K': (1.96, 0.82), 'Ca': (1.74, 1.00), 'Sc': (1.62, 1.36), 'Ti': (1.36, 1.54), 'V': (1.25, 1.63),
    'Cr': (1.27, 1.66), 'Mn': (1.39, 1.55), 'Fe': (1.25, 1.83), 'Co': (1.26, 1.88), 'Ni': (1.21, 1.91),
    'Cu': (1.38, 1.90), 'Zn': (1.31, 1.65), 'Ga': (1.26, 1.81), 'Ge': (1.22, 2.01), 'As': (1.19, 2.18),
    'Se': (1.16, 2.55), 'Br': (1.96, 2.96), 'Rb': (2.11, 0.82), 'Sr': (1.92, 0.95), 'Y': (1.62, 1.22),
    'Zr': (1.48, 1.33), 'Nb': (1.37, 1.60), 'Mo': (1.45, 2.16), 'Tc': (1.36, 1.9), 'Ru': (1.34, 2.2),
    'Rh': (1.34, 2.28), 'Pd': (1.37, 2.20), 'Ag': (1.53, 1.93), 'Cd': (1.48, 1.69), 'In': (1.44, 1.78),
    'Sn': (1.41, 1.96), 'Sb': (1.38, 2.05), 'Te': (1.35, 2.1), 'I': (2.20, 2.66), 'Cs': (2.25, 0.79),
    'Ba': (1.98, 0.89), 'La': (1.69, 1.10), 'Ce': (1.81, 1.12), 'Pr': (1.82, 1.13), 'Nd': (1.82, 1.14),
    'Pm': (1.81, 1.13), 'Sm': (1.80, 1.17), 'Eu': (1.99, 1.2), 'Gd': (1.79, 1.20), 'Tb': (1.76, 1.2),
    'Dy': (1.75, 1.22), 'Ho': (1.74, 1.23), 'Er': (1.73, 1.24), 'Tm': (1.72, 1.25), 'Yb': (1.94, 1.1),
    'Lu': (1.72, 1.27), 'Hf': (1.50, 1.3), 'Ta': (1.38, 1.5), 'W': (1.46, 2.36), 'Re': (1.37, 1.9),
    'Os': (1.35, 2.2), 'Ir': (1.36, 2.2), 'Pt': (1.39, 2.28), 'Au': (1.44, 2.54), 'Hg': (1.51, 2.0),
    'Tl': (1.70, 1.62), 'Pb': (1.75, 2.33), 'Bi': (1.55, 2.02)
}

# ==========================================
# 2. CORE UTILITIES: PARSING & DESCRIPTORS
# ==========================================
def parse_formula(formula):
    all_keys = sorted(ELEMENT_PROPS.keys(), key=len, reverse=True)
    pattern = "|".join(re.escape(k) for k in all_keys)
    matches = re.findall(f"({pattern})(\d*\.?\d*)", str(formula))
    
    composition = {}
    for el, count in matches:
        amt = float(count) if count else 1.0
        composition[el] = composition.get(el, 0) + amt
    return composition

def get_descriptors(comp):
    radii, ens = [], []
    for el, count in comp.items():
        if el in ELEMENT_PROPS:
            r, e = ELEMENT_PROPS[el]
            if r == 0 and e == 0: continue
            radii.extend([r] * int(max(1, round(count))))
            ens.extend([e] * int(max(1, round(count))))
    if not radii: return [0.0]*6
    return [np.mean(radii), np.min(radii), np.max(radii), 
            np.mean(ens), np.min(ens), np.max(ens)]

# ==========================================
# 3. DATA PREPARATION
# ==========================================
print("Loading data...")
df = pd.read_csv('INPUT DATA/perovskite2.csv')

# Filter
df = df[df['band_gap'] > 0.001].dropna(subset=['band_gap', 'formation_energy_per_atom', 'volume', 'density', 'crystal_system', 'formula_pretty', 'nsites'])

# Feature Engineering
print("Generating features from chemical formulas...")
comp_data = df['formula_pretty'].apply(parse_formula)
unique_components = sorted(list(set().union(*(c.keys() for c in comp_data))))

X_comp = pd.DataFrame([{el: c.get(el, 0) for el in unique_components} for c in comp_data]).fillna(0)
X_desc = pd.DataFrame([get_descriptors(c) for c in comp_data], 
                      columns=['r_mean', 'r_min', 'r_max', 'en_mean', 'en_min', 'en_max'])

X = pd.concat([X_comp, X_desc], axis=1)
X['nsites'] = df['nsites'].values
X_values = X.values.astype(np.float32)

# Targets
reg_cols = ['band_gap', 'formation_energy_per_atom', 'volume', 'density']
y_reg = df[reg_cols].values.astype(np.float32)
le = LabelEncoder()
y_clf = le.fit_transform(df['crystal_system'].astype(str))

# Splits and Scaling
X_train_r, X_test_r, y_reg_train_r, y_reg_test, y_clf_train_r, y_clf_test = train_test_split(
    X_values, y_reg, y_clf, test_size=0.2, random_state=42
)

sc_x, sc_y = StandardScaler(), StandardScaler()

X_train = torch.tensor(sc_x.fit_transform(X_train_r), dtype=torch.float32)
X_test = torch.tensor(sc_x.transform(X_test_r), dtype=torch.float32)
y_reg_train = torch.tensor(sc_y.fit_transform(y_reg_train_r), dtype=torch.float32)
y_clf_train = torch.tensor(y_clf_train_r, dtype=torch.long)

# ==========================================
# 4. MODEL ARCHITECTURE
# ==========================================
class ResidualBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim), nn.BatchNorm1d(dim), nn.ReLU(),
            nn.Dropout(0.1), nn.Linear(dim, dim), nn.BatchNorm1d(dim)
        )
        self.relu = nn.ReLU()
    def forward(self, x): 
        return self.relu(x + self.block(x))

class MultiTaskMaterialNet(nn.Module):
    def __init__(self, input_dim, n_classes, n_reg):
        super().__init__()
        self.body = nn.Sequential(
            nn.Linear(input_dim, 256), nn.BatchNorm1d(256), nn.ReLU(),
            ResidualBlock(256),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU()
        )
        self.head_reg = nn.Sequential(nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, n_reg))
        self.head_clf = nn.Sequential(nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, n_classes))
        
    def forward(self, x):
        feat = self.body(x)
        return self.head_reg(feat), self.head_clf(feat)

# ==========================================
# 5. TRAINING LOOP
# ==========================================
MTmodel = MultiTaskMaterialNet(X_values.shape[1], len(le.classes_), len(reg_cols))
optimizer = optim.Adam(MTmodel.parameters(), lr=0.001)

# Training components
mse_loss = nn.MSELoss()
ce_loss = nn.CrossEntropyLoss()

epochs = 1000
print(f"Training Multi-Task Model for {epochs} epochs...")

for epoch in range(epochs):
    MTmodel.train()
    optimizer.zero_grad()
    
    # Forward pass
    p_reg, p_clf = MTmodel(X_train)
    
    # Combined Loss calculation
    # We weight the regression loss higher here so it doesn't get overpowered by Classification
    loss = 2 * mse_loss(p_reg, y_reg_train) + ce_loss(p_clf, y_clf_train)
    
    # Backprop
    loss.backward()
    optimizer.step()
    
    if (epoch+1) % 100 == 0:
        print(f"Epoch {epoch+1:4d}/{epochs} | Loss: {loss.item():.4f}")

# ==========================================
# 6. EVALUATION
# ==========================================
print("\nEvaluating Model on Test Set...")
MTmodel.eval()
with torch.no_grad():
    p_reg_s, p_clf_logits = MTmodel(X_test)
    
    # Inverse transform to original scaling for metrics
    p_reg = sc_y.inverse_transform(p_reg_s.numpy())
    p_clf = torch.argmax(p_clf_logits, dim=1).numpy()

print("\n" + "="*50)
print(f"{'Target Property':<25} | {'MAE':<10} | {'R2':<10}")
print("-" * 50)
for i, name in enumerate(reg_cols):
    mae = mean_absolute_error(y_test_original := y_reg_test[:, i], p_reg[:, i])
    r2 = r2_score(y_test_original, p_reg[:, i])
    print(f"{name:<25} | {mae:<10.4f} | {r2:<10.4f}")
print("-" * 50)
print(f"{'Crystal System Accuracy':<25} | {accuracy_score(y_clf_test, p_clf):.2%}")
print("="*50)

Loading data...
Generating features from chemical formulas...
Training Multi-Task Model for 1000 epochs...
Epoch  100/1000 | Loss: 0.2827
Epoch  200/1000 | Loss: 0.1207
Epoch  300/1000 | Loss: 0.1098
Epoch  400/1000 | Loss: 0.1037
Epoch  500/1000 | Loss: 0.1001
Epoch  600/1000 | Loss: 0.0991
Epoch  700/1000 | Loss: 0.0987
Epoch  800/1000 | Loss: 0.0956
Epoch  900/1000 | Loss: 0.0977
Epoch 1000/1000 | Loss: 0.0937

Evaluating Model on Test Set...

Target Property           | MAE        | R2        
--------------------------------------------------
band_gap                  | 0.3435     | 0.8714    
formation_energy_per_atom | 0.0650     | 0.9696    
volume                    | 18.1286    | 0.9796    
density                   | 0.1755     | 0.9584    
--------------------------------------------------
Crystal System Accuracy   | 78.63%


RANDOM FOREST MODEL TRAINING

In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score

# ==========================================
# 1. ELEMENTAL & ORGANIC PROPERTY DATABASE
# ==========================================
ELEMENT_PROPS = {
    'H': (0.37, 2.20), 'Li': (1.34, 0.98), 'Be': (0.90, 1.57), 'B': (0.82, 2.04), 'C': (0.77, 2.55),
    'N': (0.75, 3.04), 'O': (1.40, 3.44), 'F': (1.33, 3.98), 'Na': (1.54, 0.93), 'Mg': (1.30, 1.31),
    'Al': (1.18, 1.61), 'Si': (1.11, 1.90), 'P': (1.06, 2.19), 'S': (1.02, 2.58), 'Cl': (1.81, 3.16),
    'K': (1.96, 0.82), 'Ca': (1.74, 1.00), 'Sc': (1.62, 1.36), 'Ti': (1.36, 1.54), 'V': (1.25, 1.63),
    'Cr': (1.27, 1.66), 'Mn': (1.39, 1.55), 'Fe': (1.25, 1.83), 'Co': (1.26, 1.88), 'Ni': (1.21, 1.91),
    'Cu': (1.38, 1.90), 'Zn': (1.31, 1.65), 'Ga': (1.26, 1.81), 'Ge': (1.22, 2.01), 'As': (1.19, 2.18),
    'Se': (1.16, 2.55), 'Br': (1.96, 2.96), 'Rb': (2.11, 0.82), 'Sr': (1.92, 0.95), 'Y': (1.62, 1.22),
    'Zr': (1.48, 1.33), 'Nb': (1.37, 1.60), 'Mo': (1.45, 2.16), 'Tc': (1.36, 1.9), 'Ru': (1.34, 2.2),
    'Rh': (1.34, 2.28), 'Pd': (1.37, 2.20), 'Ag': (1.53, 1.93), 'Cd': (1.48, 1.69), 'In': (1.44, 1.78),
    'Sn': (1.41, 1.96), 'Sb': (1.38, 2.05), 'Te': (1.35, 2.1), 'I': (2.20, 2.66), 'Cs': (2.25, 0.79),
    'Ba': (1.98, 0.89), 'La': (1.69, 1.10), 'Ce': (1.81, 1.12), 'Pr': (1.82, 1.13), 'Nd': (1.82, 1.14),
    'Pm': (1.81, 1.13), 'Sm': (1.80, 1.17), 'Eu': (1.99, 1.2), 'Gd': (1.79, 1.20), 'Tb': (1.76, 1.2),
    'Dy': (1.75, 1.22), 'Ho': (1.74, 1.23), 'Er': (1.73, 1.24), 'Tm': (1.72, 1.25), 'Yb': (1.94, 1.1),
    'Lu': (1.72, 1.27), 'Hf': (1.50, 1.3), 'Ta': (1.38, 1.5), 'W': (1.46, 2.36), 'Re': (1.37, 1.9),
    'Os': (1.35, 2.2), 'Ir': (1.36, 2.2), 'Pt': (1.39, 2.28), 'Au': (1.44, 2.54), 'Hg': (1.51, 2.0),
    'Tl': (1.70, 1.62), 'Pb': (1.75, 2.33), 'Bi': (1.55, 2.02)
}


# ==========================================
# 2. UTILITIES
# ==========================================
def parse_formula(formula):
    all_keys = sorted(ELEMENT_PROPS.keys(), key=len, reverse=True)
    pattern = "|".join(re.escape(k) for k in all_keys)
    matches = re.findall(f"({pattern})(\d*\.?\d*)", str(formula))
    composition = {}
    for el, count in matches:
        amt = float(count) if count else 1.0
        composition[el] = composition.get(el, 0) + amt
    return composition

def get_descriptors(comp):
    radii, ens = [], []
    for el, count in comp.items():
        if el in ELEMENT_PROPS:
            r, e = ELEMENT_PROPS[el]
            radii.extend([r] * int(max(1, round(count))))
            ens.extend([e] * int(max(1, round(count))))
    if not radii: return [0.0]*6
    return [np.mean(radii), np.min(radii), np.max(radii), 
            np.mean(ens), np.min(ens), np.max(ens)]

# ==========================================
# 3. DATA PREPARATION
# ==========================================
df = pd.read_csv('INPUT DATA/perovskite2.csv')

# Preprocessing
df = df[df['band_gap'] > 0.001]
reg_cols = [ 'volume', 'density']
# Include the factors in the dropna check to ensure we only train on complete data
df = df.dropna(subset=reg_cols + ['crystal_system', 'formula_pretty', 'nsites', 'Tolerance_factor', 'Octahedral_factor'])

# Normalize Volume
df['formula_atoms'] = df['formula_pretty'].apply(lambda x: sum(parse_formula(x).values()))



# Feature Engineering
comp_data = df['formula_pretty'].apply(parse_formula)
unique_components = sorted(list(set().union(*(c.keys() for c in comp_data))))

X_comp = pd.DataFrame([{el: c.get(el, 0) for el in unique_components} for c in comp_data]).fillna(0)
X_desc = pd.DataFrame([get_descriptors(c) for c in comp_data], 
                      columns=['r_mean', 'r_min', 'r_max', 'en_mean', 'en_min', 'en_max'])

# Combine features with Tolerance and Octahedral Factors
X = pd.concat([X_comp.reset_index(drop=True), X_desc.reset_index(drop=True)], axis=1)
X['nsites'] = df['nsites'].values
X['Tolerance_factor'] = df['Tolerance_factor'].values
X['Octahedral_factor'] = df['Octahedral_factor'].values
X["Bandgap"]= df["band_gap"].values
X["formation_energy_per_atom"]=df["formation_energy_per_atom"].values
X_values = X.values

y_reg = df[reg_cols].values
le = LabelEncoder()
y_clf = le.fit_transform(df['crystal_system'].astype(str))

X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = train_test_split(
    X_values, y_reg, y_clf, test_size=0.2, random_state=42
)

# ==========================================
# 4. RANDOM FOREST MODELS
# ==========================================
print("Training Random Forest Models with Geometric Factors...")

rf_reg = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_reg.fit(X_train, y_reg_train)

rf_clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_clf.fit(X_train, y_clf_train)

# ==========================================
# 5. PREDICTION & EVALUATION
# ==========================================
def predict_materialrf(formula, t_factor, mu_factor, band_gap=0.0, formation_energy=0.0):
    """
    Predicts properties using the trained Random Forest models.
    Note: band_gap and formation_energy must be provided because they were used as 
    training features for the other properties in your script.
    """
    comp = parse_formula(formula)
    if not comp: return "Invalid formula."

    # 1. Build composition vector (same order as training)
    vec_comp = np.zeros(len(unique_components))
    for el, count in comp.items():
        if el in unique_components:
            vec_comp[unique_components.index(el)] = count
    
    # 2. Build descriptors
    vec_desc = np.array(get_descriptors(comp)) # 6 features
    
    # 3. Scalar features
    nsites = sum(comp.values())
    
    # 4. Combine into a single array in the EXACT order of X.columns:
    # [unique_components, r_mean, r_min, r_max, en_mean, en_min, en_max, 
    #  nsites, Tolerance_factor, Octahedral_factor, Bandgap, formation_energy_per_atom]
    full_vec = np.concatenate([
        vec_comp, 
        vec_desc, 
        [nsites], 
        [t_factor], 
        [mu_factor], 
        [band_gap], 
        [formation_energy]
    ]).reshape(1, -1)
    
    # 5. Predict
    reg_res = rf_reg.predict(full_vec)[0]
    clf_res = rf_clf.predict(full_vec)[0]
    
    crystal_sys = le.inverse_transform([int(clf_res)])[0]
    
    results = {
        "crystal_system": crystal_sys,
        "volume": reg_res[0],
        "density": reg_res[1]
    }
    
    return results


# Full evaluation block for all attributes
p_reg = rf_reg.predict(X_test)
p_clf = rf_clf.predict(X_test)

print("\n" + "="*60)
print(f"{'Target Property':<25} | {'MAE':<12} | {'R2 Score':<10}")
print("-" * 60)
for i, name in enumerate(reg_cols):
    mae = mean_absolute_error(y_reg_test[:, i], p_reg[:, i])
    r2 = r2_score(y_reg_test[:, i], p_reg[:, i])
    print(f"{name:<25} | {mae:<12.4f} | {r2:<10.4f}")

print("-" * 60)
acc = accuracy_score(y_clf_test, p_clf)
print(f"{'Crystal System Accuracy':<25} | {acc:.2%}")
print("="*60)

# Example Usage:
result = predict_materialrf("Ba2FeWO6", 0.985, 0.482, band_gap=2.5, formation_energy=-2.1)
print(result)

Training Random Forest Models with Geometric Factors...

Target Property           | MAE          | R2 Score  
------------------------------------------------------------
volume                    | 13.6596      | 0.9767    
density                   | 0.3593       | 0.8571    
------------------------------------------------------------
Crystal System Accuracy   | 79.28%
{'crystal_system': 'Cubic', 'volume': 144.44406617400003, 'density': 6.475099619149997}


In [ ]:
ml=pd.read_csv("Generated_Data.csv")
ml["Band_Gap"]=0.0
ml["Formation_Energy"]=0.0
ml["Volume"]=0.0
ml["Density"]=0.0
ml["Crystal_System"]=""

GENERATED DATASET ATTRIBUTE PREDICTION USING THE TWO MODELS

In [9]:
#generate predicted values


import os
import re
import torch
import joblib
from pymatgen.core import Structure





# --- 2. HELPER FUNCTIONS ---
def get_nsites(formula):
    """Counts total atoms in formula for the GNN."""
    counts = re.findall(r'[0-9]+', formula)
    if not counts: return 5 # Default ABX3
    return sum(int(c) for c in counts)



# --- 3. INTEGRATED LOOP ---
print("Predicting Attributes for all data...")
count = 0


# -- PREDICTED FILE ---
ml = pd.read_csv("OUTPUT DATA/Generated_Data.csv")
n = len(ml)
BATCH_SIZE = 1000
output_file = "OUTPUT DATA/Predictions.csv"

# Instead of ml3 = pd.DataFrame, use a list to collect results
all_results = []

count = 0
for i in range(0, n, BATCH_SIZE):
    # Slice the dataframe batch
    ml_batch = ml.iloc[i : i + BATCH_SIZE].copy()
    
    # Temporarily store batch results to avoid .at[] speed issues
    batch_data = []

    for index, row in ml_batch.iterrows():
        count += 1
        if count % 100 == 0: print(f"Processing row {count}...")
        expanded_formula = row["Formula_pretty"]
        
        try:
            # Model 1: Matbench
            r1 = predict_materialMt(expanded_formula)
            density = r1["density"]
            formation_energy = r1["formation_energy_per_atom"]
            band_gap = r1["band_gap"]
            
            # Factors
            t_factor = row["Tolerance_factor"]
            o_factor = row["Octahedral_factor"]
            nsites = get_nsites(expanded_formula)

            # Model 2: RF
            r2 = predict_materialrf(expanded_formula, t_factor, o_factor)
            

            # Create a dictionary of the new results
            prediction_entry = {
                **row.to_dict(), # Keep original CSV data
                "N_Sites": nsites,
                "Crystal_System": r2["crystal_system"],
                "Formation_Energy": formation_energy,
                "Density": density,
                "Band_Gap": band_gap
            }
            batch_data.append(prediction_entry)
            
        except Exception as e:
            print(f"Skipping {expanded_formula} due to error: {e}")
            print(r1,"\n",r2)
            exit

    # Convert batch list to DataFrame and save/append
    if batch_data:
        batch_df = pd.DataFrame(batch_data)
        
        # Write to file immediately so you don't lose data if the script crashes
        if i == 0:
            batch_df.to_csv(output_file, index=False)
        else:
            batch_df.to_csv(output_file, mode='a', header=False, index=False)

print(f"Done! Predictions saved to {output_file}")

Predicting with Hybrid GNN (r3) integration...
Processing row 100...
Processing row 200...
Processing row 300...
Processing row 400...
Processing row 500...
Processing row 600...
Processing row 700...
Processing row 800...
Processing row 900...
Processing row 1000...
Processing row 1100...
Processing row 1200...
Processing row 1300...
Processing row 1400...
Processing row 1500...
Processing row 1600...
Processing row 1700...
Processing row 1800...
Processing row 1900...
Processing row 2000...
Processing row 2100...
Processing row 2200...
Processing row 2300...
Processing row 2400...
Processing row 2500...
Processing row 2600...
Processing row 2700...
Processing row 2800...
Processing row 2900...
Processing row 3000...
Processing row 3100...
Processing row 3200...
Processing row 3300...
Processing row 3400...
Processing row 3500...
Processing row 3600...
Processing row 3700...
Processing row 3800...
Processing row 3900...
Processing row 4000...
Processing row 4100...
Processing row 4200